<a href="https://colab.research.google.com/github/afethuseynzade21-prog/SQLab/blob/main/Rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install langchain langchain-community \
             chromadb sentence-transformers \
             pypdf python-docx \
             transformers accelerate -q

print("Qurulum tamamlandı!")



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/2

In [6]:
import torch

if torch.cuda.is_available():
    print(f"GPU tapıldı: {torch.cuda.get_device_name(0)}")
else:
    print(" GPU yoxdur — CPU ilə işləyəcək (yavaş olar)")

GPU tapıldı: Tesla T4


In [7]:
!pip install wikipedia -q
print(" Wikipedia paketi quruldu!")

  Preparing metadata (setup.py) ... done
 Wikipedia paketi quruldu!


In [8]:
from langchain_community.document_loaders import WikipediaLoader

print("Wikipedia-dan 'Artificial Intelligence' yüklənir...")

loader = WikipediaLoader(
    query="Artificial Intelligence",
    lang="en",
    load_max_docs=2,
)

docs = loader.load()

print(f"{len(docs)} məqalə yükləndi!")
for d in docs:
    print(f"   {d.metadata['title']}")
print(f" Ümumi simvol: {sum(len(d.page_content) for d in docs):,}")
print(f"\n--- İlk 300 simvol ---\n{docs[0].page_content[:300]}...")

Wikipedia-dan 'Artificial Intelligence' yüklənir...
2 məqalə yükləndi!
   Artificial intelligence
   Artificial general intelligence
 Ümumi simvol: 8,000

--- İlk 300 simvol ---
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develo...


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "],
)

chunks = splitter.split_documents(docs)

print(f"{len(chunks)} chunk yaradıldı!")
print(f"Ortalama chunk ölçüsü: {sum(len(c.page_content) for c in chunks) // len(chunks)} simvol")
print(f"\n--- Nümunə chunk ---")
print(chunks[0].page_content)  # 2 əvəzinə 0

41 chunk yaradıldı!
Ortalama chunk ölçüsü: 194 simvol

--- Nümunə chunk ---
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making


In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
)

# Yeni qovluq adı — köhnəsi ilə qarışmır
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db_v2",
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f" {len(chunks)} chunk saxlandı!")

/tmp/ipykernel_1280/1975162444.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 41 chunk saxlandı!


In [12]:
!pip install langchain-huggingface -q
print(" Hazır!")

 Hazır!


In [13]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device=0,
    max_new_tokens=200,
    do_sample=False,
)

llm = HuggingFacePipeline(pipeline=pipe)

# TinyLlama-nın öz chat formatı
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""<|system|>
You are a helpful assistant. Answer only based on the given context.</s>
<|user|>
Context: {context}

Question: {question}</s>
<|assistant|>"""
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Hazır!")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Hazır!


In [14]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# UI elementləri
sual_input = widgets.Text(
    placeholder="Sualınızı buraya yazın...",
    layout=widgets.Layout(width="80%")
)
gonder_btn = widgets.Button(
    description="Göndər ➤",
    button_style="primary",
    layout=widgets.Layout(width="18%")
)
output = widgets.Output()

def cavab_ver(event):
    sual = sual_input.value.strip()
    if not sual:
        return

    sual_input.value = ""  # input-u təmizlə

    with output:
        print(f"\n❓ Sual: {sual}")
        print("⏳ Cavab hazırlanır...")

        cavab = rag_chain.invoke(sual)
        if "<|assistant|>" in cavab:
            cavab = cavab.split("<|assistant|>")[-1].strip()

        print(f"\n🤖 Cavab:\n{cavab}")
        print("\n📚 Mənbələr:")
        for i, doc in enumerate(retriever.invoke(sual)):
            print(f"  [{i+1}] {doc.page_content[:150]}...")
        print("─"*50)

# Düyməyə və ya Enter-ə basanda işləsin
gonder_btn.on_click(cavab_ver)
sual_input.on_submit(cavab_ver)

# Göstər
display(HTML("<h3>💬 RAG Chatbot</h3>"))
display(widgets.HBox([sual_input, gonder_btn]))
display(output)

Output()